# SYMILE final-test evaluation

In [1]:
# @@ENV@@ — initialize TensorFlow and import the final-test evaluator
import os
import sys
from pathlib import Path

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
BASE_PATH = Path(os.environ.get("DUM_BASE", os.environ.get("ISIC24_BASE", "/mnt/disk1/home/cgiovannoni/mod-attendance")))
sys.path.insert(0, str(BASE_PATH / "src"))

import tensorflow as tf
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

import pandas as pd
from IPython.display import display
from isic24 import paths as pth
from isic24 import statistical_evaluation as se
from isic24.xai_subset import CENTRAL_SETTINGS

pth.DATASET, pth.VARIANT = "symile", "processed"
ART_PATH = pth.art_path()
OUTPUT_DIR = ART_PATH / "statest"


2026-11-04 15:09:57.706263: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1793801397.719117 2380205 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1793801397.723150 2380205 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
I0000 00:00:1793801400.431795 2380205 gpu_device.cc:2413] Ignoring visible gpu device (device: 1, name: NVIDIA GeForce GT 1030, pci bus id: 0000:65:00.0, compute capability: 6.1) with core count: 3. The minimum required count is 8. You can adjust this requirement with the env var TF_MIN_GPU_MULTIPROCESSOR_COUNT.


In [2]:
# @@PROTOCOL@@ — freeze reference and validation-selected configurations before test inference
STUDIES = {
    "vanilla": "20260927-222132",
    "fixed_scaling": "20260928-052108",
    "learnable_scaling": "20260928-171346",
    "loss_regularization": "20260928-223034",
    "controlled_fusion": "20260930-124221",
    "adaptive_gating": "20260930-211234",
}
SEEDS = [8, 47, 53, 71, 90, 101]
EXPECTED_SETTINGS = {
    "vanilla": 1, "fixed_scaling": 9, "learnable_scaling": 5,
    "loss_regularization": 5, "controlled_fusion": 1, "adaptive_gating": 5,
}
protocol = se.freeze_protocol(
    {name: ART_PATH / "runs" / name / study for name, study in STUDIES.items()},
    ART_PATH, OUTPUT_DIR, SEEDS, EXPECTED_SETTINGS, CENTRAL_SETTINGS,
    n_boot=2000, n_permutations=5000, random_seed=2026,
)
selection = pd.DataFrame(protocol["evaluations"])
display(selection.groupby(["panel", "strategy", "setting_tag", "delta"])
        .agg(n_seeds=("seed", "size"), validation_auc=("selection_val_auc", "mean")))


n_seeds  validation_auc
panel     strategy            setting_tag delta                         
reference adaptive_gating     base        0.00         6        0.792727
          controlled_fusion   base        0.00         6        0.792848
          fixed_scaling       a0p500      0.00         6        0.796090
          image_only_control  a0p000      0.00         6        0.746805
          learnable_scaling   init_a0p500 0.00         6        0.795669
          loss_regularization lam1e-03    0.00         6        0.800175
          table_only_control  a1p000      0.00         6        0.808642
          vanilla             baseline    0.00         6        0.800735
selected  adaptive_gating     ent1e-01    0.00         6        0.795987
          controlled_fusion   base        0.25         6        0.797776
          fixed_scaling       a0p875      0.00         6        0.807962
          image_only_control  a0p000      0.00         6        0.746805
          learnable_scaling   init_a0p900 0.00         6        0.805730
          loss_regularization lam1e-05    0.00         6        0.801681
          table_only_control  a1p000      0.00         6        0.808642
          vanilla             baseline    0.00         6        0.800735

In [3]:
# @@TEST_PREDICTIONS@@ — validate patient separation and predict the untouched test cohort
test_frame, predictions = se.predict_test(protocol, OUTPUT_DIR)
print(f"[TEST] {len(test_frame)} images, {test_frame.patient_id.nunique()} patients")

I0000 00:00:1793801506.158781 2380205 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22799 MB memory:  -> device: 0, name: Quadro RTX 6000, pci bus id: 0000:17:00.0, compute capability: 7.5
I0000 00:00:1793801517.971011 2380205 cuda_dnn.cc:529] Loaded cuDNN version 90300


[TEST] 1/78 adaptive_gating:base:d+0.000:s8
[TEST] 2/78 adaptive_gating:base:d+0.000:s47
[TEST] 3/78 adaptive_gating:base:d+0.000:s53
[TEST] 4/78 adaptive_gating:base:d+0.000:s71
[TEST] 5/78 adaptive_gating:base:d+0.000:s90
[TEST] 6/78 adaptive_gating:base:d+0.000:s101
[TEST] 7/78 controlled_fusion:base:d+0.000:s8
[TEST] 8/78 controlled_fusion:base:d+0.250:s8
[TEST] 9/78 controlled_fusion:base:d+0.000:s47
[TEST] 10/78 controlled_fusion:base:d+0.250:s47
[TEST] 11/78 controlled_fusion:base:d+0.000:s53
[TEST] 12/78 controlled_fusion:base:d+0.250:s53
[TEST] 13/78 controlled_fusion:base:d+0.000:s71
[TEST] 14/78 controlled_fusion:base:d+0.250:s71
[TEST] 15/78 controlled_fusion:base:d+0.000:s90
[TEST] 16/78 controlled_fusion:base:d+0.250:s90
[TEST] 17/78 controlled_fusion:base:d+0.000:s101
[TEST] 18/78 controlled_fusion:base:d+0.250:s101
[TEST] 19/78 fixed_scaling:a0p500:d+0.000:s8
[TEST] 20/78 fixed_scaling:a0p500:d+0.000:s47
[TEST] 21/78 fixed_scaling:a0p500:d+0.000:s53
[TEST] 22/78 fixed_s

In [4]:
# @@STATISTICS@@ — patient-paired confidence intervals, block-swap tests and seed sign tests
results = se.analyze_test(protocol, test_frame, predictions)

[STATISTICS] Patient bootstrap complete; computing paired contrasts
[STATISTICS] reference: 17 paired comparisons complete
[STATISTICS] selected: 17 paired comparisons complete


In [5]:
# @@RESULTS@@ — inspect predictive estimates and prespecified paired contrasts
display(results["summary"].round(5))
display(results["contrasts"].round(5))
print("Differences = strategy minus baseline: higher AUC and lower Brier/ECE are favorable.")
print("95% intervals cover patient sampling conditional on fitted seeds; seed SD is separate.")
print("Intervals are pointwise; p-values use Holm correction within each panel and test unit.")

,panel,strategy,metric,mean,seed_sd,seed_variance,ci_lo,ci_hi,n_seeds,n_images,n_patients
0,reference,adaptive_gating,auc,0.78660,0.00646,0.00004,0.75971,0.81259,6,2242,1917
1,reference,adaptive_gating,brier,0.12498,0.01153,0.00013,0.11720,0.13333,6,2242,1917
2,reference,adaptive_gating,ece,0.15174,0.03421,0.00117,0.14021,0.16368,6,2242,1917
3,reference,controlled_fusion,auc,0.78968,0.00925,0.00009,0.76365,0.81658,6,2242,1917
4,reference,controlled_fusion,brier,0.11742,0.01743,0.00030,0.10937,0.12560,6,2242,1917
5,reference,controlled_fusion,ece,0.12859,0.04559,0.00208,0.11754,0.13956,6,2242,1917
6,reference,fixed_scaling,auc,0.78666,0.01507,0.00023,0.75894,0.81344,6,2242,1917
7,reference,fixed_scaling,brier,0.13385,0.01630,0.00027,0.12655,0.14125,6,2242,1917
8,reference,fixed_scaling,ece,0.18982,0.04770,0.00228,0.17731,0.20213,6,2242,1917
9,reference,image_only_control,auc,0.74992,0.01201,0.00014,0.72086,0.77929,6,2242,1917


,panel,baseline,strategy,metric,difference,ci_lo,ci_hi,favorable_seeds,n_informative_seeds,patient_swap_p,seed_sign_p,patient_swap_p_holm,seed_sign_p_holm
0,reference,vanilla,adaptive_gating,auc,0.00616,-0.00566,0.01820,5,6,0.43671,0.21875,1.00000,1.0
1,reference,vanilla,adaptive_gating,brier,-0.01255,-0.01612,-0.00920,5,6,0.00020,0.21875,0.01020,1.0
2,reference,vanilla,adaptive_gating,ece,-0.05648,-0.06047,-0.05188,6,6,0.00020,0.03125,0.01020,1.0
3,reference,vanilla,controlled_fusion,auc,0.00923,-0.00242,0.02084,5,6,0.37173,0.21875,1.00000,1.0
4,reference,vanilla,controlled_fusion,brier,-0.02010,-0.02385,-0.01646,5,6,0.00020,0.21875,0.01020,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,selected,table_only_control,loss_regularization,brier,0.01135,0.00509,0.01785,0,6,0.00100,0.03125,0.02599,1.0
98,selected,table_only_control,loss_regularization,ece,0.01606,0.00834,0.02392,1,6,0.00040,0.21875,0.01200,1.0
99,selected,table_only_control,vanilla,auc,-0.00961,-0.03610,0.01607,1,6,0.49290,0.21875,1.00000,1.0
100,selected,table_only_control,vanilla,brier,0.00854,0.00247,0.01506,0,6,0.00740,0.03125,0.17017,1.0


Differences = strategy minus baseline: higher AUC and lower Brier/ECE are favorable.
95% intervals cover patient sampling conditional on fitted seeds; seed SD is separate.
Intervals are pointwise; p-values use Holm correction within each panel and test unit.


In [ ]:
# @@EXPORT@@ — save final-test tables, provenance and the selected-panel confidence-interval plot
se.save_results(results, protocol, OUTPUT_DIR)
print(f"[SAVED] Final-test results: {OUTPUT_DIR}")

[SAVED] Final-test results: /mnt/disk1/home/cgiovannoni/mod-attendance/artifacts/symile/processed/statest


: 